In [ ]:
from google.cloud import bigquery
from google.colab import auth

# 구글 계정 인증
auth.authenticate_user()

# BigQuery 연결
PROJECT_ID = "your-gcp-project"
client = bigquery.Client(project=PROJECT_ID)

print("BigQuery 연결 완료")

# 최종 ML Training Mart 구축

공통 BASE를 기준으로 각 담당자가 생성한 Feature Mart를 `user_id`로 결합한다.

## 결합 대상

- `mart_ml_d7_base`: 공통 모집단 및 D+7 타겟
- `feat_user_basic_d7`: 사용자 기본 정보
- `fct_ml_ping_24h`: 첫 24시간 PING 행동
- `fct_ml_hint_24h`: 첫 24시간 힌트 행동
- `fct_ml_category_24h`: 첫 24시간 질문 카테고리 행동

BASE를 기준으로 `LEFT JOIN`하여 기존 4,844명의 사용자가 유지되도록 한다.

최종 결과는 사용자 1명당 1행이며,
`retained_d7`을 머신러닝 Target으로 사용한다.

## 먼저 5개 테이블 QA

In [ ]:
qa_query = """
SELECT 'base' AS table_name,
       COUNT(*) AS row_count,
       COUNT(DISTINCT user_id) AS user_count
FROM `your-gcp-project.mart.mart_ml_d7_base`

UNION ALL

SELECT 'basic',
       COUNT(*),
       COUNT(DISTINCT user_id)
FROM `your-gcp-project.mart.feat_user_basic_d7`

UNION ALL

SELECT 'ping',
       COUNT(*),
       COUNT(DISTINCT user_id)
FROM `your-gcp-project.mart.fct_ml_ping_24h`

UNION ALL

SELECT 'hint',
       COUNT(*),
       COUNT(DISTINCT user_id)
FROM `your-gcp-project.mart.fct_ml_hint_24h`

UNION ALL

SELECT 'category',
       COUNT(*),
       COUNT(DISTINCT user_id)
FROM `your-gcp-project.mart.fct_ml_category_24h`
"""

qa_df = client.query(qa_query).to_dataframe()
qa_df

## 컬럼명 확인

In [ ]:
schema_query = """
SELECT
    table_name,
    column_name,
    data_type
FROM `your-gcp-project.mart.INFORMATION_SCHEMA.COLUMNS`

WHERE table_name IN (
    'mart_ml_d7_base',
    'feat_user_basic_d7',
    'fct_ml_ping_24h',
    'fct_ml_hint_24h',
    'fct_ml_category_24h'
)

ORDER BY table_name, ordinal_position
"""

schema_df = client.query(schema_query).to_dataframe()

display(schema_df)

## JOIN

In [ ]:
training_query = """
SELECT
    b.*,

    basic.* EXCEPT(user_id),

    p.* EXCEPT(user_id),

    h.* EXCEPT(user_id),

    c.* EXCEPT(user_id)

FROM `your-gcp-project.mart.mart_ml_d7_base` b

LEFT JOIN `your-gcp-project.mart.feat_user_basic_d7` basic
    USING(user_id)

LEFT JOIN `your-gcp-project.mart.fct_ml_ping_24h` p
    USING(user_id)

LEFT JOIN `your-gcp-project.mart.fct_ml_hint_24h` h
    USING(user_id)

LEFT JOIN `your-gcp-project.mart.fct_ml_category_24h` c
    USING(user_id)
"""

In [ ]:
training_df = client.query(training_query).to_dataframe()

print("행 수:", len(training_df))
print("고유 사용자:", training_df["user_id"].nunique())
print("중복 사용자:", training_df["user_id"].duplicated().sum())
print("컬럼 수:", len(training_df.columns))

training_df.head()

In [ ]:
for i, col in enumerate(training_df.columns, 1):
    print(f"{i:2}. {col}")

# training 생성

In [ ]:
create_training_mart_query = f"""
CREATE OR REPLACE TABLE
`your-gcp-project.mart.mart_ml_d7_training`
AS

{training_query}
"""

client.query(create_training_mart_query).result()

print("mart_ml_d7_training 생성 완료")